# CH5330 L05 v0.2 — From analytical fitting to numerical learning

**Single integrated student notebook · review candidate · not approved.**  NumPy, Matplotlib and SciPy; no GPU or online datasets. The three checkpoints can be explored at different times. Run the notebook from the top to initialise their shared functions.

**Scientific thread:** (A) one OLS minimum, three methods; (B) numerical optimisation when the logistic-regression optimum has no general closed-form formula; (C) two moons, where optimisation is successful but model representation may still be inadequate. This is the same two-moons generator and 70/15/15 seed-42 split as L06 notebook v0.3. The **test set is reserved**; only validation results are used when comparing model choices.

**Before each experiment:** predict → modify a parameter → observe → explain. Algorithm controls: starting coefficients, learning rate $\eta_{\rm lr}$, iteration count, and solver tolerance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
SEED = 42
np.set_printoptions(precision=5, suppress=True)
print('NumPy:', np.__version__)


## A — Analytical and numerical routes to the same OLS optimum

We observe $N$ pairs $(x_i,y_i)$. The line $\hat y_i=wx_i+b$ predicts $y_i$ using **slope** $w$ and **intercept** $b$. The residual is $e_i=wx_i+b-y_i$. Minimise

$$\mathcal L(w,b)=\frac1{2N}\sum_{i=1}^N e_i^2.$$

The factor $1/2$ simplifies derivatives and does not change the minimiser. The **gradient** $\nabla\mathcal L$ contains both partial derivatives; a partial derivative changes one parameter while holding the other fixed. In the traditional regression notation of L03, $w$ corresponds to slope $b_{\rm L03}$ and $b$ here corresponds to intercept $a_{\rm L03}$.

We calculate (i) the normal-equation solution, (ii) full-batch gradient descent, and (iii) a general-purpose SciPy numerical minimiser on **the exact same loss**.

In [ ]:
x_ols = np.arange(5.0)
y_ols = np.array([0.7, 2.1, 2.4, 4.2, 4.6])
N_ols = len(x_ols)

def ols_loss(theta):
    w, b = theta
    residual = w*x_ols + b - y_ols
    return np.mean(residual**2)/2

def ols_grad(theta):
    w, b = theta
    residual = w*x_ols + b - y_ols
    return np.array([np.mean(x_ols*residual), np.mean(residual)])

w_exact = np.sum((x_ols-x_ols.mean())*(y_ols-y_ols.mean())) / np.sum((x_ols-x_ols.mean())**2)
b_exact = y_ols.mean() - w_exact*x_ols.mean()
theta_exact = np.array([w_exact,b_exact])
print('Analytical normal equations:',theta_exact,'loss:',round(ols_loss(theta_exact),8))
assert np.allclose(theta_exact,[.99,.82])


### What a gradient-descent iteration actually does

At iteration $k$, calculate the error and gradient **at the current values**, then update **both parameters simultaneously**:

$$\begin{pmatrix}w_{k+1}\\b_{k+1}\end{pmatrix}=\begin{pmatrix}w_k\\b_k\end{pmatrix}-\eta_{\rm lr}\begin{pmatrix}\partial\mathcal L/\partial w\\\partial\mathcal L/\partial b\end{pmatrix}_{(w_k,b_k)}.$$

The gradient points towards *local increase* of the loss. Its negative is a descent direction for a sufficiently small positive step. **Training** means applying the rule repeatedly; it does not guarantee convergence for arbitrary learning rate. `steps` is a chosen stopping rule, not a proof of optimality.

In [ ]:
def gradient_descent_ols(start=(0.,0.), learning_rate=0.12, steps=150):
    theta = np.asarray(start,dtype=float).copy()
    history=[theta.copy()]
    for k in range(steps):
        gradient=ols_grad(theta)
        theta = theta - learning_rate*gradient  # simultaneous update
        history.append(theta.copy())
    return np.asarray(history)

# CHECKPOINT A — change these, predict how the trajectory changes.
ols_start=(0.,0.)
ols_learning_rate=0.12
ols_steps=150
history=gradient_descent_ols(ols_start,ols_learning_rate,ols_steps)

# SciPy's BFGS uses a different optimisation algorithm, but the SAME loss and gradient.
solver_tol=1e-10
solver=minimize(ols_loss,np.asarray(ols_start),jac=ols_grad,method='BFGS',
                options={'gtol':solver_tol,'maxiter':2000})
print('Gradient descent:',history[-1], 'loss:',ols_loss(history[-1]))
print('SciPy BFGS:',solver.x,'loss:',solver.fun,
      'iterations:',solver.nit,'converged:',solver.success,solver.message)
print('Difference GD from exact:',np.linalg.norm(history[-1]-theta_exact))
print('Difference solver from exact:',np.linalg.norm(solver.x-theta_exact))
assert solver.success and np.linalg.norm(solver.x-theta_exact)<1e-6


In [ ]:
ws=np.linspace(-.4,1.55,140); bs=np.linspace(-.3,1.8,140)
WW,BB=np.meshgrid(ws,bs)
LL=np.mean((WW[...,None]*x_ols+BB[...,None]-y_ols)**2,axis=2)/2
fig,axes=plt.subplots(1,2,figsize=(11,4.2))
ax=axes[0]
ax.contour(WW,BB,LL,levels=[.05,.08,.15,.3,.6,1.2,2,3.5],colors='0.6')
ax.plot(history[:,0],history[:,1],'-',color='#bd8036',lw=1.8,label='Gradient descent')
ax.scatter(*theta_exact,marker='*',s=170,c='#087980',label='Analytical')
ax.scatter(*solver.x,marker='x',s=90,c='black',label='SciPy BFGS')
ax.set(xlabel='slope w',ylabel='intercept b',title='One loss landscape')
ax.legend(fontsize=8)
ax=axes[1]
ax.plot([ols_loss(t) for t in history],label='Gradient descent')
ax.axhline(ols_loss(theta_exact),ls='--',color='#087980',label='Analytical minimum')
ax.set(xlabel='Iteration',ylabel='Half-mean-squared-error loss',title='Approach to the minimum')
ax.legend(fontsize=8)
plt.tight_layout();plt.show()


**Checkpoint A questions.** (1) Try $\eta_{\rm lr}=0.02$ and $0.5$: how does convergence change? (2) Start from $(w,b)=(1.5,-0.2)$: does the analytical optimum move? (3) Try a too-large learning rate: does the loss always decrease? (4) Why do we not compare only iterations across BFGS and gradient descent? **Numerical tolerance** is a target accuracy; a solver's convergence report should be checked alongside its residual and distance from the known minimum. In OLS, a unique analytical solution makes this unusually easy.

## B — What survives when an analytical optimum is unavailable?

For binary observations $y\in\{0,1\}$, define the **affine score** $s=\mathbf w^{\mathsf T}\mathbf x+b$ and the **sigmoid probability** $\hat p=1/(1+e^{-s})$. Fit with binary cross-entropy (BCE):

$$\mathcal L_{\rm BCE}=-\frac1N\sum_i\left[y_i\ln\hat p_i+(1-y_i)\ln(1-\hat p_i)\right].$$

Its gradient has a compact analytical formula, $\nabla_{\mathbf w}\mathcal L=N^{-1}X^{\mathsf T}(\mathbf p-\mathbf y)$ and $\partial\mathcal L/\partial b=N^{-1}\sum_i(p_i-y_i)$, but the optimum is *generally not available as a closed-form expression*. Numerical optimisation remains available. Logistic loss is convex in the fitted parameters for fixed features, although a finite optimum may fail to exist for perfectly separable, unregularised data. We use a nonseparable noisy dataset here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A fixed random seed gives the same synthetic data whenever you restart.
SEED = 42
rng = np.random.default_rng(SEED)

def make_two_moons(n_per_class=100, noise=0.16, seed=SEED):
    """Make two interlocking crescent-shaped groups of 2D points.

    This is a deliberately simple alternative to importing scikit-learn.
    The labels are 0 (upper crescent) and 1 (lower crescent).
    """
    random = np.random.default_rng(seed)
    t0 = random.uniform(0, np.pi, n_per_class)
    t1 = random.uniform(0, np.pi, n_per_class)
    upper = np.column_stack([np.cos(t0), np.sin(t0)])
    lower = np.column_stack([1 - np.cos(t1), 0.5 - np.sin(t1)])
    X = np.vstack([upper, lower])
    X += random.normal(0, noise, X.shape)
    y = np.array([0] * n_per_class + [1] * n_per_class)
    # Shuffle the data so that the two classes are interspersed.
    order = random.permutation(len(y))
    return X[order], y[order]

def split_data(X, y, seed=SEED):
    """Keep a test set separate. Only training points fit the model."""
    random = np.random.default_rng(seed + 1)
    # Split *within each class* so that small validation/test sets stay balanced.
    by_class = [random.permutation(np.where(y == label)[0]) for label in (0, 1)]
    train_ids, valid_ids, test_ids = [], [], []
    for indices in by_class:
        n_train = int(0.70 * len(indices))
        n_valid = int(0.15 * len(indices))
        train_ids.extend(indices[:n_train])
        valid_ids.extend(indices[n_train:n_train + n_valid])
        test_ids.extend(indices[n_train + n_valid:])
    return [(X[ids], y[ids]) for ids in (train_ids, valid_ids, test_ids)]

def show_points(X, y, title='Two moons'):
    plt.figure(figsize=(6, 4))
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=21, edgecolors='none')
    plt.xlabel('$x_1$'); plt.ylabel('$x_2$'); plt.title(title)
    plt.axis('equal'); plt.grid(alpha=0.15); plt.show()

X, y = make_two_moons()
(X_train, y_train), (X_val, y_val), (X_test, y_test) = split_data(X, y)
print(f'Training: {len(y_train)} | Validation: {len(y_val)} | Test: {len(y_test)}')
show_points(X_train, y_train, 'Training data (two interleaving crescents)')

In [ ]:
def sigmoid(z):
    """Logistic activation: maps any real number to (0, 1)."""
    z = np.clip(z, -40, 40)  # prevents overflow in exp for extreme inputs
    return 1 / (1 + np.exp(-z))

def binary_cross_entropy(y, probabilities):
    """Average loss for labels 0/1; clipping avoids log(0)."""
    p = np.clip(probabilities, 1e-10, 1 - 1e-10)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def accuracy(y, probabilities):
    predictions = (probabilities >= 0.5).astype(int)
    return np.mean(predictions == y)

def show_boundary(predict_probabilities, X, y, title):
    """Draw the p(class 1)=0.5 boundary of any binary classifier."""
    xmin, xmax = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    ymin, ymax = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
    xs = np.linspace(xmin, xmax, 90)
    ys = np.linspace(ymin, ymax, 90)
    xx, yy = np.meshgrid(xs, ys)
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    probs = predict_probabilities(grid).reshape(xx.shape)
    plt.figure(figsize=(6, 4))
    plt.contourf(xx, yy, probs, levels=np.linspace(0, 1, 11), cmap='coolwarm', alpha=0.32)
    plt.contour(xx, yy, probs, levels=[0.5], colors='black', linewidths=1.3)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=17, edgecolors='none')
    plt.xlabel('$x_1$'); plt.ylabel('$x_2$'); plt.title(title)
    plt.axis('equal'); plt.show()

In [ ]:
def logistic_loss(theta, X, y):
    w,b=theta[:-1],theta[-1]
    return binary_cross_entropy(y,sigmoid(X@w+b))

def logistic_grad(theta,X,y):
    w,b=theta[:-1],theta[-1]
    error=sigmoid(X@w+b)-y
    return np.r_[X.T@error/len(y),np.mean(error)]

def train_logistic_gd(X,y,learning_rate=0.5,steps=800):
    theta=np.zeros(X.shape[1]+1)
    track=[logistic_loss(theta,X,y)]
    for iteration in range(steps):
        theta -= learning_rate * logistic_grad(theta,X,y)
        track.append(logistic_loss(theta,X,y))
    return theta,np.array(track)

# CHECKPOINT B: compare an explicit training loop with a standard solver.
logistic_learning_rate=0.5
logistic_steps=800
linear_theta,logistic_track=train_logistic_gd(X_train,y_train,logistic_learning_rate,logistic_steps)
logistic_solver=minimize(logistic_loss,np.zeros(3),args=(X_train,y_train),
                         jac=logistic_grad,method='BFGS',options={'gtol':1e-8,'maxiter':500})
predict_linear=lambda X:sigmoid(X@linear_theta[:-1]+linear_theta[-1])
print('GD theta:',np.round(linear_theta,4),'validation accuracy:',accuracy(y_val,predict_linear(X_val)))
print('BFGS theta:',np.round(logistic_solver.x,4),'success:',logistic_solver.success,
      'gradient norm:',np.linalg.norm(logistic_grad(logistic_solver.x,X_train,y_train)))
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].plot(logistic_track,label='Gradient descent')
axes[0].axhline(logistic_solver.fun,ls='--',color='black',label='BFGS loss')
axes[0].set(xlabel='Iteration',ylabel='Training BCE',title='Numerical minimisation');axes[0].legend()
plt.sca(axes[1]);plt.scatter(X_train[:,0],X_train[:,1],c=y_train,cmap='coolwarm',s=16)
xx=np.linspace(X_train[:,0].min()-.2,X_train[:,0].max()+.2,130)
if abs(linear_theta[1])>1e-8:
    yy=-(linear_theta[0]*xx+linear_theta[2])/linear_theta[1]
    axes[1].plot(xx,yy,color='black',label='p = 0.5 boundary')
axes[1].set(xlabel='$x_1$',ylabel='$x_2$',title='The fitted boundary remains a line')
axes[1].legend();axes[1].set_aspect('equal',adjustable='box')
plt.tight_layout();plt.show()


**Checkpoint B questions.** Why is it legitimate to compute a gradient even when we cannot write down the optimal fitted weights directly? What does changing the learning rate change: the optimisation trajectory, the *family of possible boundaries*, or both? The solver is not a black-box answer: inspect its convergence flag and gradient norm. The **same training split** is used in both numerical algorithms.

## C — Optimisation cannot create missing expressive capacity

With $\hat p=\sigma(\mathbf w^\mathsf T\mathbf x+b)$, the threshold $\hat p=0.5$ means $\mathbf w^\mathsf T\mathbf x+b=0$: a **straight line** in two input dimensions. Better numerical optimisation cannot make this boundary bend. We can instead **change the representation** by supplying polynomial features $\boldsymbol\phi(\mathbf x)=(x_1,x_2,x_1^2,x_1x_2,x_2^2)$. The score $\mathbf w^\mathsf T\boldsymbol\phi(\mathbf x)+b$ remains affine in the fitted coefficients but may be nonlinear in the original inputs. This is not necessarily better predictive performance. Here the numerical budgets are also different; comparisons should be interpreted as demonstrations of possible boundary shapes, not as an isolated feature-effect estimate.

In [ ]:
def quadratic_features(X):
    x1,x2=X[:,0],X[:,1]
    return np.column_stack([x1,x2,x1*x1,x1*x2,x2*x2])
# Same algorithm and data split; numerical step size and iteration budget also differ.
# This is an illustrative comparison, not a fully controlled ablation.
quadratic_theta,quadratic_history=train_logistic_gd(quadratic_features(X_train),y_train,
                                                     learning_rate=0.2,steps=2000)
predict_quadratic=lambda X:sigmoid(quadratic_features(X)@quadratic_theta[:-1]+quadratic_theta[-1])
fig,axes=plt.subplots(1,2,figsize=(11,4))
xx,yy=np.meshgrid(np.linspace(-1.5,2.6,160),np.linspace(-1.2,1.6,130))
grid=np.c_[xx.ravel(),yy.ravel()]
for ax,predict,name in zip(axes,[predict_linear,predict_quadratic],['Linear classifier','Fixed quadratic features']):
    probabilities=predict(grid).reshape(xx.shape)
    ax.contourf(xx,yy,probabilities,levels=np.linspace(0,1,11),cmap='coolwarm',alpha=.25)
    ax.contour(xx,yy,probabilities,levels=[.5],colors='black')
    ax.scatter(X_train[:,0],X_train[:,1],c=y_train,cmap='coolwarm',s=13)
    ax.set(xlabel='$x_1$',ylabel='$x_2$',title=name)
    ax.set_aspect('equal',adjustable='box')
plt.tight_layout();plt.show()
print('Linear validation:',accuracy(y_val,predict_linear(X_val)))
print('Quadratic validation:',accuracy(y_val,predict_quadratic(X_val)))


**Checkpoint C questions.** (1) Which model can produce a curved decision boundary? (2) Does the quadratic model necessarily generalise better here? (3) Which dataset is appropriate for selecting the feature family? **Do not consult the test split while choosing.**

**The handoff to L06:** numerical optimisation can fit the parameters of a chosen family, but it cannot create a decision rule that the model family cannot express. L06 introduces *learned* nonlinear hidden features rather than manually supplying a polynomial basis. L07 develops gradient computation through multilayer networks.

### Terminology and notation
OLS = ordinary least squares; GD = gradient descent; BCE = binary cross-entropy; BFGS = Broyden–Fletcher–Goldfarb–Shanno optimisation method; LR or $\eta_{\rm lr}$ = learning rate; $\mathbf w$ = fitted weights; $b$ = bias/intercept; $\boldsymbol\theta$ = full parameter vector; $\nabla$ = gradient; **affine** = linear combination plus a constant; **feature** = numeric coordinate used for prediction; **capacity/expressivity** = what functions a model family can represent. Natural logarithm is $\ln$.
